# HTTP-запити та розбір html-файлів

In [1]:
from pathlib import Path
import io
import json
import pandas as pd
import sys
print(sys.platform, sys.version, 
      f'pandas version {pd.__version__}',sep='\n')

win32
3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]
pandas version 3.0.0


In [2]:
import requests
import bs4
from bs4 import BeautifulSoup as BSoup, SoupStrainer
import html5lib
print(f'requests version {requests.__version__}', 
      f'BeautifulSoup version {bs4.__version__}',  
      f'html5lib version {html5lib.__version__}',sep='\n')

requests version 2.32.5
BeautifulSoup version 4.14.3
html5lib version 1.1


Бібліотека <code>requests</code> містить засоби роботи з http-запитами.
Вона підтримує операції <code>get</code>, <code>post</code>, <code>put</code>, <code>delete</code>, хоча для веб-скрейпінгу потрібні тільки <code>get</code> та <code>post</code>.

Завантажте сторінку у веб-оглядачі, а потім спробуйте отримати її програмно.

In [3]:
url = 'https://en.wikipedia.org/wiki/World_population'
r  = requests.get(url)

Подивимось, який текст було отримано.

In [4]:
r.text

'Please set a user-agent and respect our robot policy https://w.wiki/4wJS. See also https://phabricator.wikimedia.org/T400119.\n'

Щось не те. З'ясуємо результат запиту (зазвичай 200 - ОК). Насправді, саме з результату запиту і слід було починати.

In [5]:
r.status_code

403

Маємо помилку виконання запиту.

Веб-сайти зазвичай містять банальні запобіжники від ботів.
У нашого запиту були такі заголовки.

In [6]:
r.request.headers

{'User-Agent': 'python-requests/2.32.5', 'Accept-Encoding': 'gzip, deflate, zstd', 'Accept': '*/*', 'Connection': 'keep-alive'}

З цих заголовків веб-сервер зробив для себе висновки і змінив видачу. Поведінка веб-сайту може бути різною. Може повертатись повідомлення, може повертатися порожній текст тощо.

Засобами веб-оглядача подивимось, які заголовки відправляє веб-оглядач, і підставимо їх у запит.

In [7]:
headers = {'User-Agent':
'Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:136.0) Gecko/20100101 Firefox/136.0',
            'Accept':
'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7'
          }

In [8]:
r = requests.get(url, headers=headers)  
r.status_code

200

Спрацювало!
Запам'ятаємо вивантажену сторінку у файл, щоб надалі мати можливість його не скачувати з інтернету.

In [9]:
fname = Path('prepare', 'density.html') 
url = 'https://en.wikipedia.org/wiki/World_population'
r  = requests.get(url, headers=headers)  
density_data = r.text
with open(fname, 'w', encoding='utf-8') as f:
    f.write(density_data)

In [10]:
fname = Path('prepare', 'area.html') 
url = 'https://en.wikipedia.org/wiki/List_of_European_countries_by_area'
r  = requests.get(url, headers=headers)  
area = r.text
with open(fname, 'w', encoding='utf-8') as f:
    f.write(area)

In [11]:
fname = Path('prepare', 'area.html') 
with open(fname,  encoding='utf-8') as f:
    area = f.read()

In [12]:
fname = Path('prepare', 'fknk.html') 
url = 'https://csc.knu.ua/uk/curriculum'
r  = requests.get(url)  
fknk = r.text
with open(fname, 'w', encoding='utf-8') as f:
    f.write(fknk)
r.status_code

200

fname = Path('prepare', 'fknk.html') 
with open(fname,  encoding='utf-8') as f:
    fknk = f.read()

In [13]:
fname = Path('prepare', 'fknk2.html') 
url = 'https://csc.knu.ua/media/study/normative-documents/documents.html'
r  = requests.get(url)  
fknk2 = r.text
with open(fname, 'w', encoding='utf-8') as f:
    f.write(fknk2)
r.status_code

200

In [14]:
fname = Path('prepare', 'fknk2.html') 
with open(fname,  encoding='utf-8') as f:
    fknk2 = f.read()

In [15]:
fname = Path('prepare', 'meteo.html') 
url = 'https://meteofor.com.ua/weather-kyiv-4944/10-days/'
r  = requests.get(url, headers=headers)  
meteo = r.text
with open(fname, 'w', encoding='utf-8') as f:
    f.write(meteo)
r.status_code

200

fname = Path('prepare', 'meteo.html') 
with open(fname,  encoding='utf-8') as f:
    meteo = f.read()

Відповідь сервера містить багато різної інформації.

In [16]:
r.encoding

'utf-8'

In [17]:
r.cookies

<RequestsCookieJar[Cookie(version=0, name='route-lua', value='1772403057.404.52.492099|2df660a123b8a9dcca7cb65dbda9074c', port=None, port_specified=False, domain='meteofor.com.ua', domain_specified=False, domain_initial_dot=False, path='/', path_specified=True, secure=False, expires=1772406656, discard=False, comment=None, comment_url=None, rest={'HttpOnly': None}, rfc2109=False)]>

In [18]:
for el in dir(r):
    if el.startswith('_'):continue
    print(el, end=', ')

apparent_encoding, close, connection, content, cookies, elapsed, encoding, headers, history, is_permanent_redirect, is_redirect, iter_content, iter_lines, json, links, next, ok, raise_for_status, raw, reason, request, status_code, text, url, 

### Розбір html-сторінки

Розбір html-сторінки можна виконати за допомогою бібліотеки <code>beautifulsoup4</code>.

(Встановлюємо бібліотеку <code>beautifulsoup4</code>, імпортуємо <code>bs4</code>.)

Клас <code>BeautifulSoup</code> у співпраці з парсером html реалізує навігацію по елементам html-сторінки.
Можна використовувати стандартний парсер (html.parser) або парсер <code>html5lib</code> зі сторонньої одноіменної бібліотеки.
Стандартний парсер працює швидше, сторонній парсер краще розуміє html-розмітку.

Виконаємо розбір сторінки "Навчальні плани" сайту факультету.

In [19]:
soup = BSoup(fknk, 'html5lib')     # html.parser
soup

<!DOCTYPE html>
<!--[if IE 9]> <html lang="en" class="ie9"> <![endif]--><!--[if !IE]><!--><html lang="en"><!--<![endif]--><!-- ТЕГИ --><head>
<meta charset="utf-8"/>
<title>Навчальні плани - Факультет комп'ютерних наук та кібернетики</title>
<meta content="IE=edge" http-equiv="X-UA-Compatible"/>
<meta content="width=device-width, initial-scale=1.0" name="viewport"/>
<meta content="text/html; charset=utf-8" http-equiv="Content-type"/>
<meta content="Django 1.9.13" name="generator"/>

<meta content="14.08.2017" name="revised"/>
<meta content="Антон Перепелиця, Ольга Руденко" name="author"/>
<meta content="Антон Перепелиця, Ольга Руденко" name="designer"/>
<meta content="Факультет комп'ютерних наук та кібернетики" name="copyright"/>
<meta content="index" name="robots"/>
<meta content="7 days" name="revisit-after"/>
<meta content="uk" name="language"/>
<meta content="Факультет комп'ютерних наук та кібернетики, комп'ютерні науки, інформатика, програмна інженерія, прикладна математика, систе

Метод <code>find</code> знаходить у дереві перший вузол, що відповідає умові. 

Перш за все можна задавати тег, який буде шукатися.

Знайдемо вузол з тегом <code>a</code> та подивимось на його атрибути.

In [20]:
soup.find('a').attrs

{'href': '/uk/contacts',
 'class': ['c-font-uppercase', 'c-font-bold', 'c-font-dark']}

Текст, що відповідає знайденому вузлу.

In [21]:
soup.find('a').text

'Контакти'

Перевірка наявності атрибуту

In [22]:
'class' in soup.find('a').attrs

True

In [23]:
soup.find('a').has_attr('class')

True

Пошук можна здійснювати не тільки за тегом, а за тегом та атрибутами. Можна використовувати синтаксис словника атрибутів.

In [24]:
soup.find('a', attrs={'target':'_blank'})

<a href="https://www.youtube.com/user/CyberneticsFaculty" target="_blank"><i class="icon-social-youtube"></i></a>

Пошук можна вести у піддереві з вершиною з заданим вузлом (у прикладі цей вузол отримується як результат пошуку).

In [25]:
soup.find('body').find('a', attrs={'target':'_blank'})

<a href="https://www.youtube.com/user/CyberneticsFaculty" target="_blank"><i class="icon-social-youtube"></i></a>

Для пошуку можна використовувати синтаксис з ключовими аргументами.

In [26]:
soup.body.find('a', target='_blank')

<a href="https://www.youtube.com/user/CyberneticsFaculty" target="_blank"><i class="icon-social-youtube"></i></a>

Оскільки <code>class</code> є ключовим словом мови Python, до іменем ключового аргументу слід робити <code>class_</code>.

In [27]:
soup.body.find('a', class_='_blank')

In [28]:
soup.find('a').has_attr('class')

True

Обійдемо всі гіпер-лінки і виведемо їх. Для обходу всіх вузлів використаємо метод <code>find_all</code>, що поверне список вузлів.

In [29]:
for el in soup.find_all('a'):
    if 'href' in el.attrs:
        print(el.attrs['href'], el.text.strip(), sep = ' --- ')
    else:
        print(el.text.strip(), el)

/uk/contacts --- Контакти
/uk/news --- Новини
http://csc50.knu.ua --- 
https://www.youtube.com/user/CyberneticsFaculty --- 
https://www.facebook.com/FacultyOfComputerScienceAndCybernetics --- 

/en/curriculum --- 
http://mail.unicyb.kiev.ua --- 
/uk/ --- Головна
/uk/history --- Історія факультету
/uk/administration --- Адміністрація
/uk/welcome-dean --- Привітання Декана
/uk/contacts --- Контакти
Вступ <a class="c-link dropdown-toggle" role="button">Вступ</a>
/uk/page-entrant --- Сторінка вступника
/uk/bachelor --- ОКР "Бакалавр"
/uk/master --- ОКР "Магістр"
/uk/postgraduate --- Аспірантура
/uk/doctoral --- Докторантура
Навчання <a class="c-link dropdown-toggle" role="button">Навчання</a>
/uk/schedule --- Графік навчального процесу
/uk/student-life --- Студентське життя
/uk/announcements --- Оголошення
/uk/curriculum --- Навчальні плани
/uk/programs --- Програми навчальних дисциплін
/uk/selected-subjects --- Дисципліни вибору
/uk/attestation --- Підсумкова атестація
/uk/military --- Ві

Розглянемо один з останніх обійдених вузлів дерева.

In [30]:
el

<a href="
/en/curriculum"><i class="icon-social-dribbble"></i></a>

З'ясуємо, які методи до нього застосовні.

In [31]:
for a in dir(el):
    if not a.startswith('_'): 
        print(a, end=', ')

EMPTY_ELEMENT_EVENT, END_ELEMENT_EVENT, MAIN_CONTENT_STRING_TYPES, START_ELEMENT_EVENT, STRING_ELEMENT_EVENT, append, attribute_value_list_class, attrs, can_be_empty_element, cdata_list_attributes, childGenerator, children, clear, contents, copy_self, css, decode, decode_contents, decompose, decomposed, default, descendants, encode, encode_contents, extend, extract, fetchAllPrevious, fetchNextSiblings, fetchParents, fetchPreviousSiblings, find, findAll, findAllNext, findAllPrevious, findChild, findChildren, findNext, findNextSibling, findNextSiblings, findParent, findParents, findPrevious, findPreviousSibling, findPreviousSiblings, find_all, find_all_next, find_all_previous, find_next, find_next_sibling, find_next_siblings, find_parent, find_parents, find_previous, find_previous_sibling, find_previous_siblings, format_string, formatter_for_name, get, getText, get_attribute_list, get_text, has_attr, has_key, hidden, index, insert, insert_after, insert_before, interesting_string_types, i

Можна бачити засоби, що виконують рух вниз та вгору по дереву (contents, children та parent), рух по братах (next_sibling, previous_sibling), рух по дереву в порядку обходу (next_element, previous_element) тощо.